# 第4章 优化算法 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter4/Chapter4.md]]（最速下降、线搜索、Momentum/NAG、矩阵AdaGrad、RMSProp、Adam、AdaMax、小批量）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第四至八节与学习率边界校注）
- [[课程/深度学习/课堂笔记/2026-09-21-第7节-随机优化算法讲解.md]]（第二至五节；Chapter4(1).py程序解读）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（d=5、SGD/Adam/RMSprop、1000步、初始全1）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 5、6、8；原参数下如实报告发散）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C4-03—10、A-06、07、10）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分及API范围）


## 使用方式

先运行环境单元，再逐题阅读、实现和自检。教学题允许提示与接口骨架；遇到“尚未完成”不代表通过。每题的代码需写在该题作答单元，答案在独立文件夹。学习时间不受 30 分钟限制；综合作业扩展题可分次完成。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path

# CPU 默认即可；不更改全局默认设备，也不依赖 CUDA、网络或额外数据文件。
torch.set_num_threads(1)
np.set_printoptions(precision=6, suppress=True)
torch.manual_seed(2026)
print("NumPy", np.__version__, "PyTorch", torch.__version__)


## Q401 驻点、鞍点与固定学习率的临界边界

**新编代码化，依据 Homework02 Exercise 5、Chapter4学习率例子、汇编 C4-03。**

1. 对 $s(x)=8x_1+12x_2+x_1^2-2x_2^2$ 求唯一驻点，沿两坐标微扰验证一增一减，不能把“梯度为0”当极小值。
2. 对 $q(x)=x_1^2+4x_2^2$，实现 `quadratic_gd(eta,steps)`，初值 `(1,1)`，返回包括初值的点序列。
比较学习率 `.1,.2,.25,.3` 的20步损失，保存 `output/Q401.png`（对数纵轴需正数）。打印第一步，解释振荡、临界不衰减、发散。


### 自检接口／需保存的结果

保存 `stationary401`（唯一驻点，NumPy数组，形状 `(2,)`）及 `changes401`（形状 `(2,)`）：取正微扰量 `.01`，两项分别记录从驻点沿第1、第2坐标正向微扰后，函数值相对驻点函数值的变化，不是微扰后的函数值本身。

`quadratic_gd(eta,steps)` 返回NumPy点序列，形状 `(steps+1,2)`，第0行为初值，其余行按更新次序排列。图片名称沿用题面，无须保存各学习率路径的额外全局名。


<details><summary>提示（先尝试再展开）</summary>

两个坐标的迭代乘子分别为 `1-2*eta`、`1-8*eta`；收敛要求两者绝对值都小于1。


</details>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
def saddle401(x): return 8*x[0]+12*x[1]+x[0]**2-2*x[1]**2
stationary401=np.array([-4.,3.]);e=.01
base=saddle401(stationary401)
changes401=np.array([saddle401(stationary401+[e,0])-base,saddle401(stationary401+[0,e])-base])
print('stationary point',stationary401,'coordinate perturbation changes',changes401)
def quadratic_gd(eta,steps):
    x=np.array([1.,1.]);path=[x.copy()]
    for _ in range(steps):
        x=x-eta*np.array([2*x[0],8*x[1]])
        path.append(x.copy())
    return np.array(path)
plt.figure()
for eta in [.1,.2,.25,.3]:
    p=quadratic_gd(eta,20);f=p[:,0]**2+4*p[:,1]**2
    print('eta',eta,'first',p[1],'final loss',f[-1])
    plt.semilogy(np.arange(21),np.maximum(f,1e-30),label=f'lr={eta}')
plt.xlabel('iteration');plt.ylabel('quadratic loss');plt.legend()
Path('output').mkdir(exist_ok=True);plt.savefig('output/Q401.png');plt.close()


### 解析与点评

**答案**：鞍点为 `(-4,3)`，二阶主方向曲率2、-4，微扰分别使函数增加 $e^2$、减少 $2e^2$。二次函数在两分量非零时固定步长收敛区间为 $0<\eta<1/4$；`.2` 的第二分量变号但衰减，`.25` 在第二坐标不衰减，`.3` 发散。点评：曲线绘图下限只用于对数显示，不改变优化参数或真实记录。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(stationary401,[-4,3]) and changes401[0]>0 and changes401[1]<0
assert np.allclose(quadratic_gd(.2,1)[1],[.6,-.6])
assert np.allclose(np.abs(quadratic_gd(.25,20)[:,1]),1)
assert np.linalg.norm(quadratic_gd(.3,20)[-1])>100
assert np.linalg.norm(quadratic_gd(.1,20)[-1])<.02
assert Path('output/Q401.png').stat().st_size>1000

print("自检通过")


## Q402 线搜索不是只解一个驻点方程

**Homework02 Exercise 6原数值＋新编验证。**

$f(x)=(x_1+x_2^2)^2$，初值 `(0,1)`，沿负梯度方向作精确线搜索。
手动求梯度与一元函数 $g(\eta)=f(x-\eta\nabla f(x))$；找出所有正的全局最优步长 `optimal_steps402`，而不是只报一个驻点。
用 Python 计算每个最优步长的新点与目标值，验证 `eta=5/16` 虽然是驻点却非最优。
再对 $q(x)=x_1^2+4x_2^2$ 实现 `quadratic_line_step(x)`，用闭式最优步长计算一步，核对新梯度与旧搜索方向正交。


### 自检接口／需保存的结果

- 定义 `objective402(x)`：输入形状 `(2,)` 的NumPy点，返回题面第一个目标f的标量值；保存初始点 `x402` 和该点解析梯度 `gradient402`，均为形状 `(2,)` 的NumPy数组。
- `optimal_steps402` 为全部正全局最优步长组成的一维NumPy数组，按升序排列；`middle_value402` 为沿该初始点负梯度方向、步长 `5/16` 处的目标值（标量）。
- `quadratic_line_step(x)` 按 `(new_point,eta)` 返回新点与步长，分别为形状 `(2,)` 的NumPy数组和标量；在零梯度处返回原点副本与步长0。对二次目标、初值 `[1,1]` 运行的两项结果，分别保存为 `new402,eta402`。


<details><summary>提示（先尝试再展开）</summary>

沿搜索方向内层为 `16*eta**2-10*eta+1`，平方后目标非负，内层的两个正根都达到全局最小值0。


</details>


In [ ]:
import numpy as np
def objective402(x): return (x[0]+x[1]**2)**2
x402=np.array([0.,1.]);gradient402=np.array([2.,4.])
optimal_steps402=np.array([1/8,1/2])
for eta in optimal_steps402:
    new=x402-eta*gradient402
    print('eta',eta,'point',new,'objective',objective402(new))
middle_value402=objective402(x402-(5/16)*gradient402)
print('other stationary point eta=5/16 has loss',middle_value402)
def quadratic_line_step(x):
    H=np.array([[2.,0.],[0.,8.]])
    g=H@x
    if np.sum(g*g)==0: return x.copy(),0.
    eta=(g@g)/(g@H@g)
    return x-eta*g,float(eta)
new402,eta402=quadratic_line_step(np.array([1.,1.]))
print('quadratic line-search step',new402,'eta',eta402)


### 解析与点评

**答案**：梯度 `(2,4)`；$g(\eta)=(16\eta^2-10\eta+1)^2$。两个最优步长 $1/8,1/2$ 都使目标0；$5/16$ 是中间驻点，不能仅由 $g'=0$ 宣称最小。二次函数令H为Hessian，$\eta=(g^Tg)/(g^THg)$，本例为17/130。点评：线搜索不是自动更便宜；每步求精确步长的额外代价需要与下降收益权衡。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(optimal_steps402,[.125,.5])
assert all(abs(objective402(x402-eta*gradient402))<1e-14 for eta in optimal_steps402)
assert middle_value402>0
assert abs(eta402-17/130)<1e-12
assert abs(np.array([2.,8.])@np.array([2*new402[0],8*new402[1]]))<1e-12
assert quadratic_line_step(np.zeros(2))[1]==0

print("自检通过")


## Q403 Momentum 与 NAG：先看清速度的约定

**新编，依据 Chapter4“动量法”、汇编 C4-05。**

在 $q(x)=x_1^2+4x_2^2$ 上手动实现 `momentum_run(x,eta,mu,steps,nag=False)`，返回点与速度历史（均含初始状态）。
采用课件速度约定 $v_{t}=\mu v_{t-1}-\eta g$、$x_t=x_{t-1}+v_t$；初速度0。
NAG 的梯度在 `x+mu*v` 评价。固定初值 `(1,1)`、eta=.1、mu=.5，比较前两步，不能把动量误写成直接加正梯度。


### 自检接口／需保存的结果

`momentum_run(...)` 按 `(points,velocities)` 返回两个NumPy数组，形状都为 `(steps+1,2)`；第0行分别是初始点与零速度。

对题面指定初值、eta、mu运行两步：常规动量的点历史保存为 `mom403`，速度历史保存为 `vm403`；相同条件下NAG的点历史保存为 `nag403`，三者形状均为 `(3,2)`。NAG速度历史的变量名可自行安排。


<details><summary>提示（先尝试再展开）</summary>

第一步两种方法相同；第二步NAG的预看位置是 `(.7,-.2)`，不是当前 `(.8,.2)`。


</details>


In [ ]:
import numpy as np
def momentum_run(x,eta,mu,steps,nag=False):
    x=np.asarray(x,dtype=float).copy();v=np.zeros_like(x)
    points=[x.copy()];velocities=[v.copy()]
    for _ in range(steps):
        look=x+mu*v if nag else x
        g=np.array([2*look[0],8*look[1]])
        v=mu*v-eta*g
        x=x+v
        points.append(x.copy());velocities.append(v.copy())
    return np.array(points),np.array(velocities)
mom403,vm403=momentum_run([1,1],.1,.5,2)
nag403,vn403=momentum_run([1,1],.1,.5,2,True)
print('Momentum points and velocities',mom403,vm403)
print('NAG points and velocities',nag403,vn403)


### 解析与点评

**答案**：Momentum第二步 `(.54,-.36)`，NAG第二步 `(.56,-.04)`。本实现速度已经包含负学习率；PyTorch SGD 常用保存“梯度动量缓冲”再减 `lr*buffer` 的等价参数化，不能直接把缓冲符号当成课件速度。初速度0、固定学习率、无dampening/weight_decay的经典动量可对应；NAG不同实现的初始化与代数写法须单独核对。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(mom403[1],[.8,.2]) and np.allclose(vm403[1],[-.2,-.8])
assert np.allclose(mom403[2],[.54,-.36])
assert np.allclose(nag403[2],[.56,-.04])
p,v=momentum_run([1.,1.],.1,0,2)
assert np.allclose(p,[[1,1],[.8,.2],[.64,.04]])

print("自检通过")


## Q404 AdaGrad：课件全矩阵形式不等于逐坐标实现

**新编，依据 Chapter4“AdaGrad算法”、9月16日第七节、汇编 C4-06。**

课件为 $G_t=\sum_{s=1}^tg_sg_s^T$、$x_t=x_{t-1}-\eta G_t^{-1/2}g_t$，奇异时取逆平方根的伪逆。
实现 `matrix_adagrad_step(x,g,G,eta)` 返回新点和新G。
用对称特征分解，特征值大于 `1e-12*max(1,max(abs(eigenvalues)))` 时取倒数平方根，其余置0。
**扩展API**：`np.linalg.eigh`、布尔索引；不属于命令表中要求背记的基础命令。

初值 `(1,1)`、梯度 `(3,4)`、历史零矩阵、eta=.1，输出一步结果，并与逐坐标AdaGrad的 `(0.9,0.9)` 比较。
再输入第二个非平行梯度 `(4,-3)`，验证累计G；最后对零梯度做无NaN检查。


### 自检接口／需保存的结果

`matrix_adagrad_step(...)` 按 `(xnew,Gnew)` 返回，类型均为NumPy数组，形状分别为 `(2,),(2,2)`。题面首步结果保存为 `x404,G404`；从该首步结果接着输入第二个指定梯度，结果保存为 `x2404,G2404`，形状对应不变。自检另行创建零梯度输入，其他内部特征分解变量可自定名。


<details><summary>提示（先尝试再展开）</summary>

首步G秩1；逆平方根作用于g得到 `g/||g||=(.6,.8)`，不是分别得到 `(1,1)`。


</details>


In [ ]:
import numpy as np
def matrix_inverse_root404(G):
    lam,Q=np.linalg.eigh((G+G.T)/2)
    cutoff=1e-12*max(1.,float(np.max(np.abs(lam))))
    weights=np.zeros_like(lam)
    mask=lam>cutoff
    weights[mask]=1/np.sqrt(lam[mask])
    return (Q*weights)@Q.T
def matrix_adagrad_step(x,g,G,eta):
    Gnew=G+np.outer(g,g)
    return x-eta*(matrix_inverse_root404(Gnew)@g),Gnew
x404,G404=matrix_adagrad_step(np.array([1.,1.]),np.array([3.,4.]),np.zeros((2,2)),.1)
x2404,G2404=matrix_adagrad_step(x404,np.array([4.,-3.]),G404,.1)
print('full matrix first step',x404,'diagonal first step',[.9,.9])
print('G first',G404,'G second',G2404,'x second',x2404)


### 解析与点评

**答案**：全矩阵首步 `(.94,.92)`，第二步 `(.86,.98)`；第二次累计G恰为25I。
PyTorch `torch.optim.Adagrad` 使用逐坐标平方和加eps的形式，不保存这个全矩阵，数值不应被强行要求相同；见[官方Adagrad公式](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adagrad.html)。
点评：矩阵平方根不是逐元素开根号；直接 `1/np.sqrt(G)` 既没有数学意义，又会被负的非对角元素破坏。特征值截断是浮点伪逆策略，作业原式未指定该数值阈值，须显式说明。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(x404,[.94,.92])
assert np.allclose(G404,[[9,12],[12,16]])
assert np.allclose(G2404,25*np.eye(2)) and np.allclose(x2404,[.86,.98])
xz,Gz=matrix_adagrad_step(np.array([1.,2.]),np.zeros(2),np.zeros((2,2)),.1)
assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))
assert not np.allclose(x404,[.9,.9])

print("自检通过")


## Q405 RMSProp：指数平方平均与零分母

**新编，依据 Chapter4“RMSProp算法”、Homework02 Exercise 8参数。**

实现 `rmsprop_step(x,g,v,eta=.01,gamma=.1)`，返回 `(xnew,vnew)`，严格用
$v_t=\gamma v_{t-1}+(1-\gamma)g_t^2$，$x_t=x_{t-1}-\eta g_t/\sqrt{v_t}$。
课件未给epsilon；本题不加epsilon，对 `g=0,v=0` 的通道约定更新为0（不计算0/0）。
从 `x=[1,1],v=[0,0]` 依次输入梯度 `[3,-4]`、`[0,2]`，打印两步。不要把gamma与新梯度权重弄反。
扩展API：`np.divide(...,out=...,where=...)` 用于安全零除；可用循环/掩码代替。


### 自检接口／需保存的结果

`rmsprop_step(...)` 按 `(xnew,vnew)` 返回两个NumPy数组，形状均为 `(2,)`；不传可选参数时使用题面默认的eta与gamma。首步结果保存为 `x405,v405`；将该首步点与历史量传入下一步，第二步结果保存为 `x2405,v2405`，四者形状均为 `(2,)`。不要将两步结果保存成同一可变数组的共享引用。


<details><summary>提示（先尝试再展开）</summary>

第一个平方平均为 `.9*g*g`；零梯度不一定让历史v归零，但当前参数更新仍为0。


</details>


In [ ]:
import numpy as np
def rmsprop_step(x,g,v,eta=.01,gamma=.1):
    vnew=gamma*v+(1-gamma)*g*g
    denom=np.sqrt(vnew)
    update=np.zeros_like(g,dtype=float)
    np.divide(g,denom,out=update,where=denom>0)
    return x-eta*update,vnew
x405,v405=rmsprop_step(np.ones(2),np.array([3.,-4.]),np.zeros(2))
x2405,v2405=rmsprop_step(x405,np.array([0.,2.]),v405)
print('step1',x405,v405,'step2',x2405,v2405)


### 解析与点评

**答案**：首步v=(8.1,14.4)，第二步v=(.81,5.04)。课件gamma=.1；PyTorch RMSprop把这一参数命名为alpha，默认alpha=.99，另在平方根后加eps，见[官方RMSprop说明](https://docs.pytorch.org/docs/stable/generated/torch.optim.RMSprop.html)。比较原式时不能直接用库默认值冒充同算法参数实验。这里的0/0→0是明确的零更新扩展约定，不是把原作业学习率改小。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(v405,[8.1,14.4])
assert np.allclose(x405,1-.01*np.array([1.,-1.])/np.sqrt(.9))
assert np.allclose(v2405,[.81,5.04])
assert abs(x2405[0]-x405[0])<1e-12 and abs(x2405[1]-(x405[1]-.02/np.sqrt(5.04)))<1e-12
xz,vz=rmsprop_step(np.array([1.,2.]),np.zeros(2),np.zeros(2))
assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))

print("自检通过")


## Q406 Adam 与 AdaMax：首步偏差修正逐项校准

**新编，依据 Chapter4“Adam”“AdaMax”、汇编 C4-07。**

实现 `optimizer_step(x,g,state,name,...)` 中Adam与AdaMax（也可实现全部6种，供长作业调用），时间计数从1开始，零初始状态。
Adam：$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$，$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$，
$\hat m=m_t/(1-\beta_1^t),\hat v=v_t/(1-\beta_2^t)$，更新 $-\eta\hat m/(\sqrt{\hat v}+\epsilon)$。
AdaMax：$u_t=\max(\beta_2u_{t-1},|g_t|)$，更新 $-\eta m_t/((1-\beta_1^t)u_t)$；**u没有二阶偏差修正**。
AdaMax按课件不加eps，零u通道约定零更新。

设置 `eta=.01,beta1=.9,beta2=.999,eps=.001`，用初始 `[1,1]`、首梯度 `[3,-4]`、第二梯度 `[-1,2]` 验证两步。
保存 `adam_first406,adam_second406,adam_state406` 和 `amax_first406,amax_second406,amax_state406`。
本题完整6种参考实现包含特征分解等扩展API；只做Adam/AdaMax时无须实现其余分支。

### 自检接口／需保存的结果

函数可采用完整签名 `optimizer_step(x,g,state,name,eta=.01,mu=.1,gamma=.1,beta1=.9,beta2=.999,eps=.001)`；本题自检只调用名称 `Adam,AdaMax`。返回值为更新后的NumPy点，形状 `(2,)`；传入的空字典 `state` 须就地初始化，并在后续调用中保存同一次运行的状态。

已指定的 `adam_first406,adam_second406,amax_first406,amax_second406` 都是各步点的独立数值快照，形状 `(2,)`。`adam_state406` 保存第二步完成后的状态，至少含键 `m,v`（各为形状 `(2,)` 的NumPy数组）；`amax_state406` 至少含键 `m,u`（同形状）。两个方法使用独立字典，额外计时或缓存键可自定。


<details><summary>提示（先尝试再展开）</summary>

首步修正后mhat=g、vhat=g²；Adam首步是 `-eta*g/(abs(g)+eps)`。AdaMax首步非零通道恰为 `-eta*sign(g)`。


</details>


In [ ]:
import numpy as np

def inv_sqrt_psd(G):
    # Teaching extension: symmetric eigendecomposition for Moore-Penrose inverse square root.
    eigenvalues,Q=np.linalg.eigh((G+G.T)/2)
    cutoff=1e-12*max(1.,float(np.max(np.abs(eigenvalues))))
    invroot=np.zeros_like(eigenvalues)
    mask=eigenvalues>cutoff
    invroot[mask]=1/np.sqrt(eigenvalues[mask])
    return (Q*invroot)@Q.T

def optimizer_step(x,g,state,name,eta=.01,mu=.1,gamma=.1,beta1=.9,beta2=.999,eps=.001):
    """Course formulas, t starts at 1. No torch.optim; state per run."""
    x=np.asarray(x,dtype=float);g=np.asarray(g,dtype=float)
    if not state:
        state.update(t=0,v=np.zeros_like(x),m=np.zeros_like(x),u=np.zeros_like(x),G=np.zeros((x.size,x.size)))
    state['t']+=1;t=state['t']
    if name=='GD':
        return x-eta*g
    if name=='Momentum':
        state['v']=mu*state['v']-eta*g
        return x+state['v']
    if name=='AdaGrad':
        state['G']=state['G']+np.outer(g,g)
        return x-eta*(inv_sqrt_psd(state['G'])@g)
    if name=='RMSProp':
        state['v']=gamma*state['v']+(1-gamma)*g*g
        denominator=np.sqrt(state['v'])
        update=np.zeros_like(g)
        np.divide(g,denominator,out=update,where=denominator>0)
        return x-eta*update
    if name=='Adam':
        state['m']=beta1*state['m']+(1-beta1)*g
        state['v']=beta2*state['v']+(1-beta2)*g*g
        mhat=state['m']/(1-beta1**t)
        vhat=state['v']/(1-beta2**t)
        return x-eta*mhat/(np.sqrt(vhat)+eps)
    if name=='AdaMax':
        state['m']=beta1*state['m']+(1-beta1)*g
        state['u']=np.maximum(beta2*state['u'],np.abs(g))
        update=np.zeros_like(g)
        np.divide(state['m']/(1-beta1**t),state['u'],out=update,where=state['u']>0)
        return x-eta*update
    raise ValueError('unknown optimizer')

adam_state406={};amax_state406={}
adam_first406=optimizer_step(np.ones(2),np.array([3.,-4.]),adam_state406,'Adam')
adam_second406=optimizer_step(adam_first406,np.array([-1.,2.]),adam_state406,'Adam')
amax_first406=optimizer_step(np.ones(2),np.array([3.,-4.]),amax_state406,'AdaMax')
amax_second406=optimizer_step(amax_first406,np.array([-1.,2.]),amax_state406,'AdaMax')
print('Adam steps',adam_first406,adam_second406,'state',adam_state406)
print('AdaMax steps',amax_first406,amax_second406,'state',amax_state406)


### 解析与点评

**答案**：Adam首步约 `(.9900033322,1.0099975006)`；AdaMax首步 `(.99,1.01)`。第2步同时检查状态和参数，能抓出只在首步“看起来正确”的指数或计时错误。
Adam的eps在**平方根外**；AdaMax课件只修正一阶矩。PyTorch Adam同类公式默认eps通常不同，本作业必须明确设为.001，见[官方Adam公式](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)；PyTorch Adamax采用 `u=max(beta2*u,abs(g)+eps)`，非零通道也与无eps课件原式存在微小差异，见[官方Adamax公式](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adamax.html)。它们不是同名即逐位相同。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert np.allclose(adam_first406,1-.01*np.array([3.,-4.])/(np.array([3.,4.])+.001))
assert np.allclose(amax_first406,[.99,1.01])
m2=.9*(.1*np.array([3.,-4.]))+.1*np.array([-1.,2.])
v2=.999*(.001*np.array([9.,16.]))+.001*np.array([1.,4.])
u2=np.maximum(.999*np.array([3.,4.]),[1.,2.])
assert np.allclose(adam_state406['m'],m2) and np.allclose(adam_state406['v'],v2)
assert np.allclose(adam_second406,adam_first406-.01*(m2/(1-.9**2))/(np.sqrt(v2/(1-.999**2))+.001))
assert np.allclose(amax_state406['u'],u2)
assert np.allclose(amax_second406,amax_first406-.01*m2/((1-.9**2)*u2))
for name in ['Adam','AdaMax']:
    xz=optimizer_step(np.array([1.,2.]),np.zeros(2),{},name)
    assert np.allclose(xz,[1,2]) and np.all(np.isfinite(xz))

print("自检通过")


## Q407 复现 Chapter4(1).py：三优化器的完整数值日志

**原课堂演示独立复现，非新造教师参数。**

保留 `d=5`、初始全1、学习率.01、1000次更新，分别用SGD、Adam、RMSprop。
目标用Python零起始下标写为
$F(x)=\frac1{d-3}\sum_{i=1}^{d-3}(x_{i-1}x_i+x_ix_{i+1}+x_{i+1}x_{i+2}-i/(d-3))^2$。
实现 `demo_objective(x)` 与 `run_demo_optimizer(name)`，返回最终参数、1001个**更新后重算**的目标值。
每种方法必须重建参数和优化器，保存 `demo_results407`，画半对数曲线 `output/Q407.png`。

本练习为数值校准将张量显式设为float64（原脚本默认float32），其它设置不改；库算法参数采用各自默认值。
教学扩展：`torch.optim.RMSprop` 不在考试命令表核心两优化器之列；本题为课堂覆盖练习，不强加到测试。
说明：这里没有任何mini-batch采样；优化器名叫SGD不代表这个程序使用随机梯度。

### 自检接口／需保存的结果

`demo_objective(x)` 接收一维PyTorch张量并返回可反传的标量张量。`run_demo_optimizer(name)` 按 `(final_x,history)` 返回两个NumPy数组，形状分别为 `(5,),(1001,)`，history包含初值及1000次更新后的目标值。

`demo_results407` 保存为字典，键准确使用 `SGD,Adam,RMSprop`，每个值均为该方法返回的二元组；日志应是纯数值，不是计算图对象。图片文件名沿用题面。


<details><summary>提示（先尝试再展开）</summary>

初值每个括号分别是2.5、2，故初始目标是41/8。目标中i=1时实际访问x的0—3号分量。


</details>


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
def demo_objective(x):
    d=x.numel();total=x.sum()*0
    for i in range(1,d-2):
        total=total+(x[i-1]*x[i]+x[i]*x[i+1]+x[i+1]*x[i+2]-i/(d-3))**2
    return total/(d-3)
def run_demo_optimizer(name):
    x=torch.tensor([1.]*5,dtype=torch.float64,requires_grad=True)
    classes={'SGD':torch.optim.SGD,'Adam':torch.optim.Adam,'RMSprop':torch.optim.RMSprop}
    optimizer=classes[name]([x],lr=.01)
    values=[demo_objective(x).item()]
    for _ in range(1000):
        optimizer.zero_grad()
        loss=demo_objective(x)
        loss.backward();optimizer.step()
        values.append(demo_objective(x).item())
    return x.detach().numpy().copy(),np.array(values)
demo_results407={name:run_demo_optimizer(name) for name in ['SGD','Adam','RMSprop']}
plt.figure()
for name,(x,history) in demo_results407.items():
    print(name,'initial',history[0],'final',history[-1],'x',x)
    plt.semilogy(np.arange(1001),np.maximum(history,1e-30),label=name)
plt.xlabel('iteration');plt.ylabel('objective');plt.legend()
Path('output').mkdir(exist_ok=True);plt.savefig('output/Q407.png');plt.close()


### 解析与点评

**答案要点**：每条日志有1001个数，首项41/8，末项必须与最终参数下重算目标吻合。原示例中的同名 `fseq` 会被后一个方法覆盖，本题用字典独立保存。点评：半对数绘图的1e-30下限只用于显示，真实记录不裁剪；不能从一次曲线给出优化器普遍排名；float64适配使结果可能与原float32末位不同。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
for name,(x,history) in demo_results407.items():
    assert history.shape==(1001,) and abs(history[0]-41/8)<1e-12
    assert np.all(np.isfinite(history)) and history[-1]<history[0]
    assert abs(history[-1]-demo_objective(torch.tensor(x)).item())<1e-12
probe=torch.tensor([1.]*5,dtype=torch.float64,requires_grad=True)
demo_objective(probe).backward()
assert np.allclose(probe.grad.numpy(),[2.5,7.,9.,6.5,2.])
assert Path('output/Q407.png').stat().st_size>1000

print("自检通过")


## Q408 小批量：末批、加权平均与真正的更新

**新编，依据9月21日第二节、Chapter4小批量、汇编 C4-08。**

数据 `x=linspace(-1,1,11)`、`y=2*x-.5`；线性模型 $a x+b$，损失MSE。
实现 `batch_gradient(theta,x,y)` 和 `make_batches(n,size,rng)`；批量大小4，打乱一次后必须覆盖所有11条且无重复，末批3条。
在**相同参数** `theta=[0,0]` 下，验证“各批平均梯度按批大小加权”恰等全梯度；未经加权的三批平均一般不等。
再做60轮小批量SGD（eta=.1），每轮重新打乱，保存 `theta408,losses408,steps408`（含初值的轮末全损失）。
不能声称更新中的三批梯度还可直接平均为同一个参数点的全梯度。
只需命令表的随机均匀数与 `np.argsort` 即可打乱，不要求DataLoader。


### 自检接口／需保存的结果

- `batch_gradient(theta,x,y)` 返回形状 `(2,)` 的NumPy平均梯度；`make_batches(n,size,rng)` 返回索引数组的list，每个元素是一批样本的整数索引。
- 将 `make_batches(11,4,np.random.default_rng(408))` 得到的首轮划分保存为 `batches408`，三批大小应依次为4、4、3。在同一参数 `[0,0]` 下，分别保存全数据梯度 `full_gradient408`、按批大小加权的批梯度平均 `weighted408`、未经加权的三批梯度平均 `unweighted408`，三者均为形状 `(2,)` 的NumPy数组；这些验证量须在参数更新前计算。
- 训练部分保存最终参数 `theta408`（NumPy数组，形状 `(2,)`）、轮末全损失 `losses408`（NumPy数组，形状 `(61,)`，含初始损失）、实际更新次数 `steps408`（整数）。训练打乱可另建seed=1408的生成器，避免首轮验证消耗的随机数改变训练实验。


<details><summary>提示（先尝试再展开）</summary>

最后一批较小时，等权平均批梯度相当于过度加权该批样本。一个epoch为3次更新，60轮是180步。


</details>


In [ ]:
import numpy as np
def batch_gradient(theta,x,y):
    r=theta[0]*x+theta[1]-y
    return np.array([2*np.mean(r*x),2*np.mean(r)])
def make_batches(n,size,rng):
    order=np.argsort(rng.uniform(0,1,n))
    return [order[start:start+size] for start in range(0,n,size)]
x408=np.linspace(-1,1,11);y408=2*x408-.5
batches408=make_batches(11,4,np.random.default_rng(408))
full_gradient408=batch_gradient(np.zeros(2),x408,y408)
weighted408=sum(len(ix)*batch_gradient(np.zeros(2),x408[ix],y408[ix]) for ix in batches408)/11
unweighted408=sum(batch_gradient(np.zeros(2),x408[ix],y408[ix]) for ix in batches408)/3
print('batch sizes',[len(ix) for ix in batches408],'full/weighted/unweighted',full_gradient408,weighted408,unweighted408)
theta408=np.zeros(2);losses408=[np.mean((theta408[0]*x408+theta408[1]-y408)**2)];steps408=0
rng408=np.random.default_rng(1408)
for epoch in range(60):
    for ix in make_batches(11,4,rng408):
        theta408-=.1*batch_gradient(theta408,x408[ix],y408[ix])
        steps408+=1
    losses408.append(np.mean((theta408[0]*x408+theta408[1]-y408)**2))
losses408=np.array(losses408)
print('final theta',theta408,'steps',steps408,'loss',losses408[-1])


### 解析与点评

**答案**：初始全梯度 `(-1.6,1)`；加权批平均完全吻合。最后参数约 `(2,-.5)`。点评：固定参数下的梯度分解是代数恒等式；训练每批后参数会变，不能据此认为mini-batch与full-batch的每轮结果完全相同。`np.sort`、`concatenate` 仅用于公开自检，学生打乱实现无需这些扩展。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert [len(ix) for ix in batches408]==[4,4,3]
assert np.array_equal(np.sort(np.concatenate(batches408)),np.arange(11))
assert np.allclose(full_gradient408,[-1.6,1.]) and np.allclose(weighted408,full_gradient408)
assert not np.allclose(unweighted408,full_gradient408)
assert steps408==180 and losses408.shape==(61,) and losses408[-1]<1e-9
assert np.max(np.abs(theta408-[2.,-.5]))<1e-4

print("自检通过")


## Q409 Homework02 Exercise 8：六手动优化器×四初值，诚实记录发散

**原作业核心编程任务完整覆盖，长作业扩展，建议60—90分钟；不是30分钟测试。**

$f(x)=100(x_2-x_1^2)^2+(1-x_1)^2$，全局极小点 `(1,1)`、最小值0。
手动实现GD、Momentum、**课件全矩阵AdaGrad**、RMSProp、Adam、AdaMax，禁止调用torch.optim代替。
四个初值依次 `(1,0),(0,1),(-1,-1),(3,3)`；每组预算1000步。**保留原作业参数**：

| 方法 | 参数 |
|---|---|
| GD | eta=.01 |
| Momentum | mu=.1, eta=.01；速度已含负学习率，v0=0 |
| AdaGrad | eta=.01；G0=0，全矩阵逆平方根伪逆 |
| RMSProp | gamma=.1, eta=.01，v0=0 |
| Adam | beta1=.9,beta2=.999,eta=.01,eps=.001，两个偏差修正 |
| AdaMax | beta1=.9,beta2=.999,eta=.01，只有一阶矩偏差修正 |

课件RMSProp/AdaMax未给eps，采用零分母通道零更新；AdaGrad按Q404的特征阈值策略。
这几个**数值实现约定**须注明，不能偷偷改eta或使用梯度裁剪后宣称“原设定收敛”。

编写 `rosenbrock`、`rosenbrock_grad`、`run_rosenbrock(name,x0,steps=1000)`；字典结果至少含
`status,stop_step,completed_steps,last_finite_x,final_x,distance,final_loss,losses,points`。
若出现非有限梯度/新点/目标，立即停止该次运行，标为发散，`final_x=None`、距离和最终目标记inf，
不能把最后有限点冒充1000步答案。正常走满预算标记 `completed_budget`，**这不等于收敛**。
保存24个结果于 `results409`，输出状态表、真实迭代数、最终点/最后有限点、距 `(1,1)` 的误差；
每个初值画两个视图：前12步的6条有效前缀，以及走满预算方法的1000步曲线，写入 `output/Q409.png`。
这样既保留发散到非有限前的真实大值，又不会让它把自适应方法的低损失变化压成一条线；完整数值仍以上述表格为准。

扩展API：`np.linalg.eigh,isfinite,errstate,divide(where=...)`、`plt.subplots`，均为长作业数值保护/显示工具，不在测试中要求默写。
另做六种算法各自的**一步数值校准**；不能只验证曲线形状。


### 自检接口／需保存的结果

`rosenbrock(x)` 返回标量目标值，`rosenbrock_grad(x)` 返回形状 `(2,)` 的NumPy梯度；输入x为长度2的NumPy点。`results409` 为含24个字典的list，每个字典对应一种方法与一个初值组合；`run_rosenbrock(...)` 的返回字典使用以下完整结构：

| 键 | 类型与含义 |
|---|---|
| `method` | 方法名字符串，使用 `GD,Momentum,AdaGrad,RMSProp,Adam,AdaMax` 之一 |
| `initial` | 初始点的独立NumPy数组，形状 `(2,)` |
| `status` | 正常用满预算为 `completed_budget`；非有限退出用以 `diverged` 开头的原因字符串 |
| `stop_step` | 正常完成为 `None`；发散时为首次失败尝试的步号，采用1起始计数 |
| `completed_steps` | 已成功接受的有限更新次数，整数；不包含失败的那次尝试 |
| `last_finite_x` | 最后一个有限点，NumPy数组，形状 `(2,)` |
| `final_x` | 正常完成时为最终点的NumPy数组，形状 `(2,)`；发散时为 `None` |
| `distance` | 正常完成时最终点到题面参考解的欧氏距离；发散时为浮点inf |
| `final_loss` | 正常完成时在最终点重算的目标值；发散时为浮点inf |
| `losses` | NumPy数值数组，形状 `(completed_steps+1,)`；第0项是初始目标，仅保留有限值 |
| `points` | NumPy数组，形状 `(completed_steps+1,2)`；第0行为初始点，逐行对应losses |

另保存字典 `one_step409`，键为上述六个方法名，每个值为形状 `(2,)` 的一步更新后NumPy点。此一步校准统一使用起点 `[1,1]`、给定梯度 `[3,-4]`、每种方法独立的零初始状态，以及题面表格中的原超参数；这里给定的是梯度输入，不再把它解释为Rosenbrock在该点的梯度。其余绘图、状态缓存变量可自定。


<details><summary>提示（先尝试再展开）</summary>

先校准梯度 `(1,0)→(400,-200)`：GD原步长首步立即到 `(-3,2)`，目标会增大。发散是应报告的数值结果，不一定是公式编错。


</details>


In [ ]:
import numpy as np

def inv_sqrt_psd(G):
    # Teaching extension: symmetric eigendecomposition for Moore-Penrose inverse square root.
    eigenvalues,Q=np.linalg.eigh((G+G.T)/2)
    cutoff=1e-12*max(1.,float(np.max(np.abs(eigenvalues))))
    invroot=np.zeros_like(eigenvalues)
    mask=eigenvalues>cutoff
    invroot[mask]=1/np.sqrt(eigenvalues[mask])
    return (Q*invroot)@Q.T

def optimizer_step(x,g,state,name,eta=.01,mu=.1,gamma=.1,beta1=.9,beta2=.999,eps=.001):
    """Course formulas, t starts at 1. No torch.optim; state per run."""
    x=np.asarray(x,dtype=float);g=np.asarray(g,dtype=float)
    if not state:
        state.update(t=0,v=np.zeros_like(x),m=np.zeros_like(x),u=np.zeros_like(x),G=np.zeros((x.size,x.size)))
    state['t']+=1;t=state['t']
    if name=='GD':
        return x-eta*g
    if name=='Momentum':
        state['v']=mu*state['v']-eta*g
        return x+state['v']
    if name=='AdaGrad':
        state['G']=state['G']+np.outer(g,g)
        return x-eta*(inv_sqrt_psd(state['G'])@g)
    if name=='RMSProp':
        state['v']=gamma*state['v']+(1-gamma)*g*g
        denominator=np.sqrt(state['v'])
        update=np.zeros_like(g)
        np.divide(g,denominator,out=update,where=denominator>0)
        return x-eta*update
    if name=='Adam':
        state['m']=beta1*state['m']+(1-beta1)*g
        state['v']=beta2*state['v']+(1-beta2)*g*g
        mhat=state['m']/(1-beta1**t)
        vhat=state['v']/(1-beta2**t)
        return x-eta*mhat/(np.sqrt(vhat)+eps)
    if name=='AdaMax':
        state['m']=beta1*state['m']+(1-beta1)*g
        state['u']=np.maximum(beta2*state['u'],np.abs(g))
        update=np.zeros_like(g)
        np.divide(state['m']/(1-beta1**t),state['u'],out=update,where=state['u']>0)
        return x-eta*update
    raise ValueError('unknown optimizer')

import numpy as np
def rosenbrock(x):
    r=x[1]-x[0]**2
    return 100*r*r+(1-x[0])**2
def rosenbrock_grad(x):
    r=x[1]-x[0]**2
    return np.array([-400*x[0]*r+2*(x[0]-1),200*r])

def run_rosenbrock(name,x0,steps=1000):
    x=np.array(x0,dtype=float);state={}
    losses=[float(rosenbrock(x))];points=[x.copy()]
    status='completed_budget';stop_step=None
    for t in range(1,steps+1):
        with np.errstate(over='ignore',invalid='ignore',divide='ignore'):
            g=rosenbrock_grad(x)
            if not np.all(np.isfinite(g)):
                status='diverged_nonfinite_gradient';stop_step=t;break
            new=optimizer_step(x,g,state,name,eta=.01,mu=.1,gamma=.1,beta1=.9,beta2=.999,eps=.001)
            new_loss=rosenbrock(new)
        if not np.all(np.isfinite(new)) or not np.isfinite(new_loss):
            status='diverged_nonfinite_update_or_loss';stop_step=t;break
        x=new;points.append(x.copy());losses.append(float(new_loss))
    complete=(status=='completed_budget')
    return dict(method=name,initial=np.array(x0),status=status,stop_step=stop_step,
                completed_steps=len(losses)-1,last_finite_x=x.copy(),
                final_x=x.copy() if complete else None,
                distance=float(np.linalg.norm(x-np.ones(2))) if complete else float('inf'),
                final_loss=float(losses[-1]) if complete else float('inf'),
                losses=np.array(losses),points=np.array(points))

import matplotlib.pyplot as plt
from pathlib import Path
methods409=['GD','Momentum','AdaGrad','RMSProp','Adam','AdaMax']
starts409=[(1,0),(0,1),(-1,-1),(3,3)]
results409=[run_rosenbrock(name,start) for start in starts409 for name in methods409]
fig,axes=plt.subplots(4,2,figsize=(12,14))
for row,start in enumerate(starts409):
    ax,ax_long=axes[row]
    for r in [r for r in results409 if np.array_equal(r['initial'],start)]:
        suffix=' [diverged]' if r['final_x'] is None else ''
        prefix=r['losses'][:13]
        ax.semilogy(np.arange(len(prefix)),np.maximum(prefix,1e-30),label=r['method']+suffix)
        if r['final_x'] is not None:
            ax_long.semilogy(np.arange(len(r['losses'])),np.maximum(r['losses'],1e-30),label=r['method'])
        print('initial',start,'method',r['method'],'status',r['status'],'stop step',r['stop_step'],
              'completed steps',r['completed_steps'],'final x',r['final_x'],
              'last finite x',r['last_finite_x'],'distance',r['distance'],'final loss',r['final_loss'])
    ax.set_title(f'initial={start}; first 12 updates');ax.set_xlabel('iteration');ax.set_ylabel('Rosenbrock loss');ax.legend(fontsize=7)
    ax_long.set_title(f'initial={start}; completed budget');ax_long.set_xlabel('iteration');ax_long.set_ylabel('Rosenbrock loss');ax_long.legend(fontsize=7)
fig.tight_layout();Path('output').mkdir(exist_ok=True);fig.savefig('output/Q409.png');plt.close(fig)
print('nonfinite runs:',sum(r['final_x'] is None for r in results409),'of',len(results409))
# Each method gets an independent one-step known-value oracle.
one_step409={name:optimizer_step(np.ones(2),np.array([3.,-4.]),{},name) for name in methods409}
print('one-step checks',one_step409)


### 解析与点评

**答案要点**：6×4=24次运行，保留原学习率和全部超参数；观察到的GD/动量非有限必须记录退出步数。自适应方法走满1000步只说明完成预算，距 `(1,1)` 仍可能很大，不能等同收敛。

**版本点评**：AdaGrad采用课件全矩阵形式；若另实现逐坐标版，须另名并单独画，不能替换本作业主结果。AdaGrad阈值是浮点伪逆容差，RMSProp/AdaMax零更新约定只处理原式未定义的0/0；没有eps或学习率的隐性改动。Adam的eps=.001按原作业。图只显示发散前有效前缀，状态表才保存是否走满预算的关键事实。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
assert rosenbrock(np.ones(2))==0 and np.allclose(rosenbrock_grad(np.ones(2)),[0,0])
assert np.allclose(rosenbrock_grad(np.array([1.,0.])),[400,-200])
probe=np.array([-.7,.4]);h=1e-6
fd=np.array([(rosenbrock(probe+h*np.eye(2)[j])-rosenbrock(probe-h*np.eye(2)[j]))/(2*h) for j in range(2)])
assert np.max(np.abs(fd-rosenbrock_grad(probe)))<1e-5
expected={
 'GD':np.array([.97,1.04]),'Momentum':np.array([.97,1.04]),
 'AdaGrad':np.array([.994,1.008]),
 'RMSProp':1-.01*np.array([1.,-1.])/np.sqrt(.9),
 'Adam':1-.01*np.array([3.,-4.])/(np.array([3.,4.])+.001),
 'AdaMax':np.array([.99,1.01])}
assert all(np.allclose(one_step409[name],target,atol=1e-12) for name,target in expected.items())
assert len(results409)==24
assert any(r['final_x'] is None for r in results409 if r['method']=='GD')
for r in results409:
    assert len(r['losses'])==r['completed_steps']+1 and r['points'].shape==(len(r['losses']),2)
    assert np.all(np.isfinite(r['losses']))
    assert np.allclose(r['points'][0],r['initial'])
    assert abs(r['losses'][0]-rosenbrock(r['initial']))<1e-10
    if r['final_x'] is None:
        assert r['status'].startswith('diverged') and r['completed_steps']<1000
        assert np.isinf(r['distance']) and np.isinf(r['final_loss'])
    else:
        assert r['completed_steps']==1000 and r['status']=='completed_budget'
        assert abs(r['distance']-np.linalg.norm(r['final_x']-1))<1e-10
        assert abs(r['final_loss']-rosenbrock(r['final_x']))<1e-10
assert Path('output/Q409.png').stat().st_size>1000

print("自检通过")


## Q410 梯度连乘与可控的稳定优化实验

**新编，依据 Chapter4梯度消失/爆炸、汇编 C4-09。**

1. 全部预激活为正、宽度1的ReLU链，所有隐藏及输出权重均为a、偏置0，输入x=1，隐藏层数L。
用自动微分实现 `chain_gradient(a,L)` 计算输出对输入的导数，比较 a=.5、1、2 与L=1,5,10,20，验证为 $a^{L+1}$，不用图像代替数值断言。
2. 另用稳定新编目标 $J(w)=(w_1-2)^2+2(w_2+1)^2$、初始 `(0,0)`，SGD学习率.1，100步，保存 `stable_history410,stable_w410`。
打印初末损失与最终参数，说明这不是将Rosenbrock原参数改小后冒称原作业收敛。

### 自检接口／需保存的结果

`chain_gradient(a,L)` 返回普通数值，含义为题面链式网络输出对标量输入的导数。`stable_w410` 保留100次更新后的CPU float64张量，形状 `(2,)`，使自检可读取其 `detach().numpy()`；`stable_history410` 为含101个普通数值的list或一维NumPy数组，依次保存初始损失及每次更新后的损失。其它局部变量无需固定命名。


<details><summary>提示（先尝试再展开）</summary>

所有ReLU输入为正，局部导数均为1，所以只剩权重连乘；注意总权重有L+1个而非L个。


</details>


In [ ]:
import numpy as np
import torch
def chain_gradient(a,L):
    x=torch.tensor(1.,dtype=torch.float64,requires_grad=True)
    h=x
    for _ in range(L):
        z=a*h
        h=torch.maximum(z,torch.tensor(0.,dtype=torch.float64))
    y=a*h;y.backward()
    return x.grad.item()
for a in [.5,1.,2.]:
    print('weight',a,[(L,chain_gradient(a,L)) for L in [1,5,10,20]])
stable_w410=torch.tensor([0.,0.],dtype=torch.float64,requires_grad=True)
opt410=torch.optim.SGD([stable_w410],lr=.1)
def stable_loss410(): return (stable_w410[0]-2)**2+2*(stable_w410[1]+1)**2
stable_history410=[stable_loss410().item()]
for _ in range(100):
    opt410.zero_grad();stable_loss410().backward();opt410.step()
    stable_history410.append(stable_loss410().item())
print('new stable problem initial/final',stable_history410[0],stable_history410[-1],'w',stable_w410.detach())


### 解析与点评

**答案**：a=.5时L=20导数为约4.768×10⁻⁷；a=2时为2097152；说明连乘带来的衰减/放大。稳定目标的Hessian为diag(2,4)，eta=.1使两个迭代乘子为.8、.6，因而收敛。点评：梯度裁剪等工程手段可另学，但本章长作业必须先如实呈现原始算法的发散，不能用保护性修改掩盖现象。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
for a in [.5,1.,2.]:
    for L in [1,5,10,20]:
        assert abs(chain_gradient(a,L)-a**(L+1))<1e-12
assert len(stable_history410)==101 and stable_history410[0]==6
assert stable_history410[-1]<1e-16
assert np.max(np.abs(stable_w410.detach().numpy()-[2.,-1.]))<1e-8

print("自检通过")
